# RaceAI_var1.0 — 当日レース予測 実行フロー

仕様書: `docs/specs/2026-07-04-today-prediction-design.md`

**市場情報（オッズ・人気）は当日予測でも一切使わない。** 当日データ取得フローには
単勝オッズ取得が付随するが、特徴量化・推論には使わない（today_adapter.py で明示的に drop）。

このノートブックは Step 1〜8 の関数をセルに分けて呼び出す実行フローである。
JV-Link 接続（実機 Windows・32bit Python・JRA-VAN 契約回線）が必要なセルには
明示的に **「このセルの実行には JV-Link 接続環境が必要です」** と記載する。
実装フェーズ（本ノートブック作成時点）ではその接続ができないため、
当該セルはまだ実行していない。ユーザーが後日、実機環境で実行すること。


In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "common").is_dir() and (cand / "main").is_dir():
            return cand
    raise RuntimeError(f"プロジェクトルートが見つかりません: {start}")


_root = find_project_root(Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("project root:", _root)


project root: C:\Users\syugo\AI\RaceAI_var1.0


In [2]:
%load_ext autoreload
%autoreload 2
from main.notebook_bootstrap import *


## Step 0: 事前準備（当日朝など）

HC/WC（坂路調教・ウッドチップ調教）の当年分を再取得し、前処理を更新する。
当日予測には直近の調教データ（trn_hc_*/trn_wc_* 12列）が必須だが、
`run_today_se_ra_and_realtime()` は RA/SE + 速報 + オッズのみを取得し
HC/WC はカバーしない（仕様書 2-7節）。

**⚠️ このセルの実行には JV-Link 接続環境が必要です。**


In [3]:
#refresh_today_training_data()


## Step 1: 当日データ取得

`run_today_se_ra_and_realtime()` を実行し、当日（または指定日）の RA/SE を
`main/data/race/race_ra.csv` / `race_se.csv` に保存する。
続けて速報取得（天候・馬場状態・馬体重）・0B31 単勝オッズ取得（race_se.csv の
odds 列を更新するが、後続の特徴量化ステップで明示的に drop する）を行う。

**⚠️ このセルの実行には JV-Link 接続環境が必要です。**


In [4]:
race_day = None  # None = 本日。特定日を指定する場合は "20260704" のように渡す
run_today_se_ra_and_realtime(race_day)


WE v2 snapshot generated: {"date": "20260704", "events": 5, "courses": 3, "events_path": "C:\\Users\\syugo\\AI\\RaceAI_var1.0\\common\\data\\output\\realtime_we_v2\\we_events_20260704_20260704_211939.csv", "snapshot_path": "C:\\Users\\syugo\\AI\\RaceAI_var1.0\\common\\data\\output\\realtime_we_v2\\we_course_snapshot_20260704_20260704_211939.csv"}


CompletedProcess(args=['py', '-3-32', '-c', "import os, sys\nsys.path.insert(0, 'C:\\\\Users\\\\syugo\\\\AI\\\\RaceAI_var1.0')\nos.chdir('C:\\\\Users\\\\syugo\\\\AI\\\\RaceAI_var1.0')\nfrom common.data.src.get_data import run_today_se_ra_and_realtime_merge; run_today_se_ra_and_realtime_merge(dual_pass_se_then_ra=True, target_kubun='both')"], returncode=0, stdout='=== get_race_data: 予測対象レースデータ取得 ===\n取得期間: 20260704000000 -> 20260704235959\nJVOpen From 補正: 20260703000000 (単日指定時の JVOpen -1 回避)\n保存先: C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\data\\race\nOption: 1 (通常データ)\n\n--- RACEデータ取得中 (RA, SE) ---\n\n--- データ保存中 ---\nRA: 72件, SE: 942件\nフィルタ除外: 0件 / 総取得: 1014件\n重複置換(優先kubun採用): 0件\n  saved: 72 rows -> C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\data\\race\\race_ra.csv\nSaved RA: C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\data\\race\\race_ra.csv (72 records)\n  saved: 942 rows -> C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\data\\race\\race_se.csv\nSaved SE: C:\\Users\\syugo\\AI\\RaceAI_var

## Step 2: 当日データ → 既存前処理スキーマへの変換

`pure_rank/src/today_adapter.py` の `build_today_merged()` で
`race_ra.csv` / `race_se.csv`（JV-Link 生データ）を
`SE_preprocessed.parquet` / `RA_preprocessed.parquet` と同一スキーマに変換し、
SK（血統）を結合する。オッズ・人気列はこの時点で明示的に drop される。

JV-Link 接続は不要（Step 1 で保存済みの CSV を読むだけ）だが、
Step 1 が未実行の環境では `race_ra.csv` / `race_se.csv` が存在せず
`FileNotFoundError` になる。


In [5]:
from today_adapter import build_today_merged
from pathlib import Path

cfg = load_config()
race_dir = PROJECT_ROOT / "main" / "data" / "race"
preprocessed_dir = PROJECT_ROOT / cfg["data"]["preprocessed_dir"]

today_merged = build_today_merged(race_dir, preprocessed_dir)
today_merged.head()


  [today_adapter] loaded race_ra.csv: 72 rows, 46 cols
  [today_adapter] loaded race_se.csv: 942 rows, 51 cols
  [today_adapter:RA][WARN] track_condition_code=0 の行が 65 件 （障害 or 馬場状態未確定の可能性）。該当レースはシナリオ生成をスキップすること: ['2026070402010701', '2026070402010702', '2026070402010703', '2026070402010704', '2026070402010705', '2026070402010706', '2026070402010707', '2026070402010708', '2026070402010709', '2026070402010710', '2026070402010711', '2026070402010712', '2026070403020301', '2026070403020302', '2026070403020303', '2026070403020304', '2026070403020305', '2026070403020306', '2026070403020307', '2026070403020309', '2026070403020312', '2026070410020301', '2026070410020302', '2026070410020303', '2026070410020304', '2026070410020305', '2026070410020306', '2026070410020308', '2026070410020310', '2026070502010801', '2026070502010802', '2026070502010803', '2026070502010804', '2026070502010805', '2026070502010806', '2026070502010807', '2026070502010808', '2026070502010809', '2026070502010810', '20260

,race_id,year,month_day,course_code,kai,nichi,race_num,wakuban,horse_num,ketto_num,sex_code,age,trainer_code,jockey_code,burden_weight,blinker_code,horse_weight,horse_weight_change,abnormal_code,finish_rank,racetime,time_3f_after,corner_1,corner_2,corner_3,corner_4,hon_shokin,fuka_shokin,running_style_code,mining_predicted_rank,is_win,is_place,grade_code,distance,track_code,horse_count,weather_code,surface_code,track_condition_code,surface_condition,distance_category,race_date,sire_id,bms_id
0,2026070402010701,2026,704,2,1,7,1,1,1,2024106810,2,2,1114,1192,55.0,0,430.0,-10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-04,1120002625,1120002307
1,2026070402010701,2026,704,2,1,7,1,2,2,2024100255,2,2,1157,1157,55.0,0,464.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-04,1120002618,1120002121
2,2026070402010701,2026,704,2,1,7,1,3,3,2024103505,2,2,1147,1170,55.0,0,430.0,-2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-04,NaN,NaN
3,2026070402010701,2026,704,2,1,7,1,4,4,2024107000,2,2,1153,1206,52.0,0,424.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-04,1120002595,1120002219
4,2026070402010701,2026,704,2,1,7,1,5,5,2024106857,2,2,1008,1197,55.0,0,452.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-04,NaN,NaN


## Step 3-4: 当日特徴量の生成（4シナリオ: 良・稍重・重・不良）

`pure_rank/src/predict_today.py` の `build_today_features()` で
`create_features.py` と全く同じ計算式を通し、132列特徴量を
`track_condition_code` シナリオ（1=良, 2=稍重, 3=重, 4=不良）ごとに生成する。
129列は共通、3列（`hist_same_condition_win_rate` 等）のみシナリオごとに異なる。

JV-Link 接続不要（Step 2 の出力 `today_merged` と既存の
`01_preprocessed/*.parquet` のみで完結）。


In [6]:
today_features = build_today_features(today_merged, cfg)
{code: df.shape for code, df in today_features.items()}


  SE: 553,887 rows, 33 cols
  RA: 39,570 rows, 27 cols
  SK: 60,648 rows, 3 cols
  Merged: 553,887 rows, 44 cols
  Filter applied: 553,887 → 536,776 rows (removed 17,111)
  [build_today_features] Combined: 537,718 rows (hist=536,776, today=942)
  track_code value_counts (rows):
    track_code=10: 4,456
    track_code=11: 72,290
    track_code=12: 11,470
    track_code=17: 114,125
    track_code=18: 46,817
    track_code=20: 24
    track_code=23: 97,669
    track_code=24: 174,499
    track_code=52: 4,646
    track_code=54: 7,875
    track_code=55: 259
    track_code=56: 3,512
    track_code=57: 76
  course_is_small=1 (中間変数): 123,198 rows (22.9%)


C:\Users\syugo\AI\RaceAI_var1.0\pure_rank\src\predict_today.py:260: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([hist_df, today_placeholder], ignore_index=True, sort=False)


  HC: 5,113,087 rows
  WC: 741,428 rows
  [build_today_features] シナリオ間 127 列（132 - 5）の完全一致を確認 (PASS)
  [build_today_features] 特徴量列数=106


{1: (942, 132), 2: (942, 132), 3: (942, 132), 4: (942, 132)}

## Step 5: 推論（アンサンブル予測）

`run_today_predictions()` で 15 モデル（3fold×5seed）アンサンブルにより
`pred_score` / `pred_rank` / `pred_softmax_prob`（参考列）を算出する。
JV-Link 接続不要。


In [7]:
today_with_preds = run_today_predictions(today_features, cfg)
today_with_preds[1].sort_values(["race_id", "pred_rank"]).head(20)


  Loaded: lambdarank_fold1_seed42.txt
  Loaded: lambdarank_fold1_seed43.txt
  Loaded: lambdarank_fold1_seed44.txt
  Loaded: lambdarank_fold1_seed45.txt
  Loaded: lambdarank_fold1_seed46.txt
  Loaded: lambdarank_fold2_seed42.txt
  Loaded: lambdarank_fold2_seed43.txt
  Loaded: lambdarank_fold2_seed44.txt
  Loaded: lambdarank_fold2_seed45.txt
  Loaded: lambdarank_fold2_seed46.txt
  Loaded: lambdarank_fold3_seed42.txt
  Loaded: lambdarank_fold3_seed43.txt
  Loaded: lambdarank_fold3_seed44.txt
  Loaded: lambdarank_fold3_seed45.txt
  Loaded: lambdarank_fold3_seed46.txt


,race_id,year,month_day,course_code,kai,nichi,race_num,wakuban,horse_num,ketto_num,sex_code,age,trainer_code,jockey_code,burden_weight,blinker_code,horse_weight,horse_weight_change,abnormal_code,finish_rank,racetime,time_3f_after,corner_1,corner_2,corner_3,corner_4,hon_shokin,fuka_shokin,running_style_code,is_win,is_place,grade_code,distance,track_code,horse_count,weather_code,surface_code,distance_category,race_date,sire_id,bms_id,hist_last_rank,hist_avg_rank_3,hist_avg_rank_5,hist_win_rate,hist_place_rate,hist_last_last3f,hist_avg_last3f_3,hist_avg_last3f_5,hist_last_time_dev,hist_avg_time_dev_3,hist_avg_time_dev_5,hist_same_surface_win_rate,hist_same_course_win_rate,hist_same_dist_win_rate,hist_days_since_last,hist_weight_change,hist_total_prize,hist_avg_prize_3,hist_same_weather_win_rate,hist_same_weather_avg_rank,hist_same_course_dist_win_rate,hist_same_grade_win_rate,hist_top_grade_exp_count,hist_exact_dist_win_rate,hist_best_grade_ever,hist_grade_diff,hist_avg_rank_top_grade,hist_front_running_pref,season_sex_score,wakuban_surface,front_pref_x_small,field_avg_win_rate,field_avg_prize,win_rate_vs_field,prize_vs_field,hist_sire_win_rate_ts,hist_sire_surface_win_rate_ts,hist_sire_dist_win_rate_ts,hist_sire_dist_diff,hist_bms_win_rate_ts,hist_jockey_win_rate_cum,hist_jockey_win_rate_30d,hist_jockey_place_rate_30d,hist_jockey_win_rate_60d,hist_jockey_course_win_rate,hist_trainer_win_rate_cum,hist_trainer_win_rate_30d,hist_trainer_win_rate_60d,hist_trainer_surface_win_rate,hist_jockey_surface_win_rate_ts,hist_trainer_course_win_rate_ts,hist_speed_idx_last,hist_speed_idx_best,hist_speed_idx_avg3,hist_speed_idx_cond_best,field_z_time_dev,field_z_prize,field_z_last3f,field_z_win_rate,field_z_speed_idx,field_z_place_rate,field_front_runner_density,relative_post_position,trn_hc_last_3f_sec,trn_hc_last_4f_sec,trn_hc_last_200_sec,trn_hc_best_3f_14d,trn_hc_best_200_14d,trn_hc_count_14d,trn_wc_last_3f_sec,trn_wc_last_4f_sec,trn_wc_last_1f_sec,trn_wc_best_3f_14d,trn_wc_best_1f_14d,trn_wc_count_14d,trn_total_count_14d,trn_hc_rank_3f,trn_hc_rank_200,trn_hc_zscore_3f,trn_wc_rank_3f,trn_wc_zscore_3f,trn_hc_3f_delta,trn_hc_200_delta,trn_wc_3f_delta,trn_count_delta,hist_same_condition_win_rate,hist_surface_condition_win_rate,hist_best_time_same_cond,track_condition_code,surface_condition,lr_label,pred_score,pred_rank,pred_softmax_prob
798,2026070402010701,2026,704,2,1,7,1,2,2,2024100255,2,2,1157,1157,55.0,0,464.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-04,1120002618,1120002121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.999074,2.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.053775,0.078844,NaN,NaN,0.078431,0.101176,0.119451,NaN,0.000000,0.128909,0.076845,0.093333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,11,NaN,-0.280281,1,0.193273
914,2026070402010701,2026,704,2,1,7,1,4,4,2024107000,2,2,1153,1206,52.0,0,424.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-04,1120002595,1120002219,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.999074,4.0,NaN,0.0,NaN,NaN,NaN,0.066298,0.093750,0.080882,77.272727,0.072917,0.072103,NaN,NaN,0.151515,0.080882,0.084299,NaN,0.117647,0.076577,0.057252,0.082474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,11,NaN,-0.317512,2,0.184033
837,2026070402010701,2026,704,2,1,7,1,3,3,2024103505,2,2,1147,1170,55.0,0,430.0,-2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.999074,3.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.123141,NaN,NaN,0.161290,0.1485

## Step 6: 出力（フォルダ構造・CSV書き出し）

```
main/predictions/{YYYYMMDD}/{開催場所名}/{良|稍重|重|不良}/
    race_{race_id}_pred.csv
    summary.csv
```

出力前に市場情報混入チェック（`FORBIDDEN_MARKET_COLS` との突合）を実行し、
通過しなければ CSV を書き出さずエラー停止する。JV-Link 接続不要。


In [8]:
out_root = PROJECT_ROOT / "main" / "predictions"
write_predictions(today_with_preds, out_root)


  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\函館\良: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\福島\良: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\小倉\良: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\函館\稍重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\福島\稍重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\小倉\稍重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\函館\重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\福島\重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\小倉\重: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260704\函館\不良: 24 races
  [write_predictions] C:\Users\syugo\AI\RaceAI

## Step 7 (参考): 回帰テストの再実行

`pure_rank/src/regression_test_today.py` は既存の完了済みレース1件を
「未実施レース」として模擬し、当日予測パイプラインの出力が
既存オフライン評価パイプライン（`evaluate.py`）のスコアと完全一致することを
検証する。JV-Link 接続不要（`01_preprocessed/*.parquet` のみで完結）。
実装フェーズで実行済みだが、コード変更後の健全性確認に再利用できる。


In [9]:
import subprocess, syssubprocess.run([sys.executable, str(PROJECT_ROOT / "pure_rank" / "src" / "regression_test_today.py")])


SyntaxError: invalid syntax (3562603943.py, line 1)

## 市場情報混入チェック（全ステップ完了後に必ず実行）

```bash
grep -rn "odds\|popularity\|ninki\|market_log_odds\|init_score" pure_rank/src/ main/ --include="*.py"
```
